# W2-D2: RCA

Notebook n?y d?ng cluster t? D1 ?? rank root cause b?ng service graph, timestamp v? alert count. Em c?ng d?ng official incident history retrieval ?? l?y incident t??ng t?.

## 1. Load input

In [1]:
from pathlib import Path
import json
from rca import load_json, load_jsonl, build_directed_graph, run_rca

BASE = Path.cwd()
D1 = BASE.parent / 'd1'
cluster_summary = load_json(D1 / 'results' / 'cluster_summary.json')
alerts = load_jsonl(BASE / 'dataset' / 'alerts_sample.jsonl')
graph = build_directed_graph(BASE / 'dataset' / 'services.json')
history = load_json(BASE / 'dataset' / 'incidents_history.json')['incidents']
print('clusters:', len(cluster_summary['clusters']))
print('alerts:', len(alerts))
print('history:', len(history))

clusters: 3
alerts: 20
history: 29


## 2. Run RCA

In [2]:
output = run_rca(cluster_summary['clusters'], alerts, graph, history)
print(json.dumps({
    'clusters_analyzed': output['clusters_analyzed'],
    'root_causes': [item['root_cause'] for item in output['results']],
}, indent=2))

{
  "clusters_analyzed": 3,
  "root_causes": [
    "payment-svc",
    "recommender-svc",
    "search-svc"
  ]
}


## 3. Inspect top candidates

In [3]:
for item in output['results']:
    print(
        f"{item['cluster_id']} | root={item['root_cause']} | "
        f"class={item['class']} | confidence={item['confidence']} | "
        f"top3={item['graph_top3'][:3]}"
    )

c-000-000 | root=payment-svc | class=connection_pool_exhaustion | confidence=0.7508 | top3=[['payment-svc', 0.9025], ['checkout-svc', 0.6587], ['cart-svc', 0.6213]]
c-000-001 | root=recommender-svc | class=batch_overlap | confidence=0.7833 | top3=[['recommender-svc', 1.0]]
c-000-002 | root=search-svc | class=n_plus_1 | confidence=0.7833 | top3=[['search-svc', 1.0]]


## 4. Write output

In [4]:
out_path = BASE / 'results' / 'rca_output.json'
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(output, ensure_ascii=False, indent=2), encoding='utf-8')
print(out_path)
print('valid:', json.loads(out_path.read_text(encoding='utf-8'))['clusters_analyzed'])

D:\Study\Home_work\xBrain\aiops-dinhdanhnam-AIO1\w2\d2\results\rca_output.json
valid: 3
